# Phase 03 — Standardisation and quality checks

**Project:** Open Materials Data Pipeline for Hydrogel Bioink Design
**Source:** [Zenodo record 19602891](https://zenodo.org/records/19602891), version v0.0.2

**What this notebook does.** It applies the Phase 02 schema. Every source cell is first reconciled against the original workbook, then standardised only where the evidence supports it: explicit percentage strain is divided by 100, other supported quantities keep their recorded units, and the `mPas` viscosity token is kept without conversion. Image dates gain an ISO-format copy; blank annotations stay unknown. Phase 02's QC findings are linked to the exact cells and image records they concern, and a new review flag marks shear rates below the published method range. No value is deleted, clipped, imputed or averaged.

| | |
|---|---|
| **Inputs** | The two source files; the 11 Phase 01 tables and manifest; the 26 Phase 02 tables and manifest |
| **Outputs** | 13 tables (CSV) and `standardisation_run_manifest.json` in `data/processed/standardisation_and_quality_checks/`; `phase03_checks.csv` in the same folder |
| **Runtime** | About 1–2 minutes (every source cell is re-read from the workbooks) |
| **Requires** | Phase 02 run in this project folder (12 of 12 checks passed) |

**How to run.** Select the kernel built from `environment.yml`, then **Kernel → Restart Kernel and Run All Cells**. Section 10 reports this phase's checks as **PASS**, **WARN** or **FAIL**; any failure stops the notebook with the reason. Section 11 lists the findings with the status labels defined in Phase 00: **Verified**, **Preliminary**, **Assumption**, **Unresolved**.

**Scope.** Standardisation here means consistent fields, units and statuses, not statistical scaling. No curve fitting, linear-viscoelastic-region selection, record linkage or modelling is performed. Worksheets and image records are not treated as confirmed physical samples.

## 1. Settings and helper functions

Paths follow the fixed repository layout, so the notebook gives the same result whether it is opened from the repository root or from `notebooks/`. `check` adds a row to the structural ledger exported as `validation_summary.csv` and stops the notebook on failure. `phase_check` adds a row to this phase's summary checks (section 10). The reference checksums identify the Phase 02 tables this step was designed and tested against.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import csv
import hashlib
import io
import json
import platform
import re
import zipfile

import numpy as np
import openpyxl
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)

EXPECTED_SOURCE_RECORD = "https://zenodo.org/records/19602891"
EXPECTED_SOURCE_VERSION = "v0.0.2"
EXPECTED_PHASE1_NAME = "audit_run_manifest.json"
EXPECTED_PHASE2_NAME = "schema_qc_run_manifest.json"
RAW_REL = Path("data/raw/zenodo_19602891")
P1_REL = Path("data/metadata/zenodo_source_audit")
P2_REL = Path("data/metadata/schema_and_quality_control")
OUT_REL = Path("data/processed/standardisation_and_quality_checks")

PHASE02_REFERENCE_SHA256 = {
    "input_verification.csv": "25a7af10c26835e88d2d43314532b07ae53040fea38637d3444d8d03bc5da05c",
    "reviewed_inputs.csv": "61188081050fd080792c106f667800d136fe94b975c6c1250c8980d27da41f21",
    "literature_evidence_register.csv": "01be2f6d3cb0e59740b631da534e7bbefb7a8d5f4744c156b9711dcfaabf824f",
    "entity_schema.csv": "b9901b9b5f3f73a486973b9f9a064e61001d47c57579e40b352aa5d4c88d7eaa",
    "deferred_entities.csv": "d25b05900e44a959a7f762a66c3c918f4e20341f3a067ef72cf83489f4824572",
    "source_to_schema_mapping.csv": "9545f539a2ff545771d5c2a0e6f6946846800a79d755cebfbf39d34a80910750",
    "image_field_dictionary.csv": "dcf5b68b8c15cb209559d27729fd5a45fc36fcef24f1e1179607bb148c9a962b",
    "data_dictionary.csv": "4d3985143ca73b5c7ea242ae940320233a5cb66657c0ec68ae8e4e412a734d2e",
    "missing_value_rules.csv": "e0c235fafc123a0d853baf0733472c8328e5249bfe2e12c106c7249eb200ebf3",
    "worksheet_register.csv": "55cf6681264c510d30a9c027d03f5dd536d3cea934b579935fd272372f683995",
    "measurement_points.csv": "4f690ae7fddc9b4cca5294250859c2fafabb0927dcf0dfdad79a609e7f8ec78f",
    "measurement_values_source.csv": "2f495ed1545a61e88a5ef181a15e97ea67e4ba4b5ebebfb8c2029fc8f6e52687",
    "image_records_source.csv": "003a0a0ffb7f807646332aaafc14ab08b4f9d627e5b82bdceba440b23f1f5fc4",
    "image_missingness.csv": "d1dc372cd1b51c3a185b97db35e36dfbf277dd1d72f997c584ca14089a5e7ec2",
    "image_observed_categories.csv": "e2fbe1568b7d0043c5750f6fbfa05fdb52fccd34afde73b1af978db879460666",
    "quality_control_rules.csv": "31d70daddbe1ebcece094bbb4286cef8615e4fdda53b80fd9bccedb156024357",
    "validation_summary.csv": "4a438638c9151df082f2e03ef8817df07e7f502a6a5473b5ba51ba8e1b1b45a9",
    "value_qc_findings.csv": "469e1c663d1bf6d03c9a7f515fb50daf8f387331e678e265335e0dd38a67a9a1",
    "strain_conversion_preview.csv": "dab0b045756c37055fd7993f4c22eed5c832969f62a7ecfd27ffa5db25a5bfb1",
    "measurement_context_summary.csv": "9aa021f4e0b63a1b035063cf5f734a036bb21e3bdb2b4dcb3cb509a055a98c45",
    "context_review.csv": "27f95971c44a719d85345413b86ce890ca3515ab30281a4d1278970cde22df64",
    "phase1_issue_carry_forward.csv": "1b898b430040b1ea0e5cc76362923d3f7bc1d830e49d5eb60ef8223da4f6013e",
    "phase2_issue_decisions.csv": "b3ad54ad613940fdcb60e168c7eeb21c49c1df4dc50de012a5884f5ebce5a580",
    "additional_review_items.csv": "c26e2ceede6b2af554617c0e5132749c11dc55213fa612f9daca5291bd941406",
    "filename_candidates_unconfirmed.csv": "a3f95d08e9393b7e871d9a2103b85f2c0306bb46357d36b8ac03cd6496e4003e",
    "linkage_summary.csv": "3d36ab09fb6e12714d59f3028b6ef187e589e3b2cc35fd3786536d0b6f582510",
}
REFERENCE_EXPORT_SHA256 = {
    "input_verification.csv": "530d2a5009e9b7469de9d56e27bfc6f68084c79b3a443e038b046b1c64d18dda",
    "document_evidence_register.csv": "5ed7547798c863b264f6b3b99853843cb0f9d595c54cf89d622a52d0cfc7e417",
    "standardisation_decisions.csv": "15d901df91effdce9e015d8e2262116bf5191c9b01fdc404b834c2436cf964eb",
    "phase3_issue_decisions.csv": "0727cb683d2708217207ca8479aa8810aa94be9bbe07382a87c1a60d7dc0b4aa",
    "transformation_rules.csv": "1fbb881cb80aeec054b75eda0fab58487c0f5c0835ce940c9a41b44e94e91145",
    "measurement_values_standardised.csv": "99a68fa0618692fb146d67007e34854868ac36e20c22602db75362af6c89d5c3",
    "measurement_points_view.csv": "d4ae41c1be16bf019f7f06b8e6294d35ad5fad17f160f56f75973d984f1b4411",
    "image_records_standardised.csv": "30a464a9e1c185a76c8443dc56eb8bfa6c43973f9b5cd231071ef6a2d8303207",
    "qc_findings.csv": "6fc139a84ab5ad234538d3c3f203f56ddedd2e4e02dea50f1d3023a22c4737f6",
    "qc_observation_links.csv": "428052749c6f73b694a60391a7831bd2d3a69d405e49e9f877ad2587e504b58b",
    "acquisition_context_coverage.csv": "ce5c2ba1fd5c17ce88f54f48d41bfe7587d18e6bd200adca7caf63df37e32c4a",
    "unresolved_review_items.csv": "217d869d621ea1bbfdd27400cd7848208ed63b241ed1d4051d62ea41b8f21107",
    "validation_summary.csv": "89d4af1f3dc5a53262d36861e3c929640c4aaaf2ea7bdf3c69694fe62e4df9ed",
}
# Tables that record checksums of earlier run manifests, which contain run times. They differ on
# every fresh run of Phases 01-02, so they are saved but left out of the reference comparison.
RUN_SPECIFIC_TABLES = {"input_verification.csv", "validation_summary.csv"}
EXPECTED = {"ledger_checks": 674, "cells": 39100, "points": 4600, "images": 1031,
            "percent_strain": 100, "fraction_strain": 2200, "qc_findings": 25, "exports": 13}

checks = []         # structural ledger, exported as validation_summary.csv
phase_checks = []   # this phase's summary checks (section 10)

def check(name, condition, evidence=""):
    ok = bool(condition)
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "evidence": str(evidence)})
    if not ok:
        raise ValueError(f"{name}: {evidence}. Investigate before continuing; do not bypass this check.")

def phase_check(check_id, description, passed, detail, warn_only=False):
    status = "PASS" if passed else ("WARN" if warn_only else "FAIL")
    phase_checks.append({"check_id": check_id, "description": description, "status": status, "detail": detail})
    print(f"[{status:4}] {check_id}: {detail}")
    if status == "FAIL":
        raise RuntimeError(f"{check_id} failed: {detail}")

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False, encoding="utf-8-sig")

here = Path.cwd().resolve()
project_root = next((p for p in (here, *here.parents)
                     if (p / "README.md").is_file() and (p / "environment.yml").is_file() and (p / "notebooks").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Project folder not found. Open this notebook from inside the hydrogel-bioink-data-curation repository.")
raw_dir, p1_dir, p2_dir = project_root / RAW_REL, project_root / P1_REL, project_root / P2_REL
OUT = project_root / OUT_REL
manifest1_path, manifest2_path = p1_dir / EXPECTED_PHASE1_NAME, p2_dir / EXPECTED_PHASE2_NAME
phase_check("P03-01", "Project folder located", True, f"repository root found ('{project_root.name}')")

missing = [p.relative_to(project_root).as_posix() for p in [manifest1_path, manifest2_path] if not p.is_file()]
phase_check("P03-02", "Phase 01 and Phase 02 manifests present", not missing,
       "present" if not missing else "missing: " + ", ".join(missing) + " — run Phases 01 and 02 first")
m2 = json.loads(manifest2_path.read_text(encoding="utf-8-sig"))
observed_manifest_hash = sha(manifest2_path)

runtime = pd.DataFrame([{"component": "Python", "version": platform.python_version()}] +
                       [{"component": name, "version": version(name)} for name in ["pandas", "numpy", "openpyxl", "ipython"]])
print("Settings loaded. Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__)

[PASS] P03-01: repository root found ('hydrogel-github-main')
[PASS] P03-02: present
Settings loaded. Python 3.12.16 | pandas 2.2.3 | numpy 2.5.3


## 2. Verify the Phase 01 and Phase 02 handoff

Both manifests are checked for the expected phase, source record and version. Every file they list (two source files, 11 Phase 01 tables, 26 Phase 02 tables) must exist at its standard location and match its recorded SHA-256; the Phase 01 manifest must match the checksum Phase 02 recorded for it. Row counts, core counts and the exact column list of every table are then checked. The Phase 02 tables must also match the reference checksums from section 1.

The Phase 02 manifest is not compared byte for byte, because it contains its run time; its checksum is recorded as `RECORDED_NOT_PREPINNED`.

In [2]:
required_manifest2_keys = {"phase", "source_record", "source_version", "phase1_manifest_sha256",
                           "verified_input_sha256", "rows", "export_sha256", "environment"}
check("Phase 2 manifest required keys", required_manifest2_keys.issubset(m2), sorted(required_manifest2_keys - set(m2)))
check("Phase 2 manifest phase identity", m2["phase"] == "2: schema and QC prototype", m2["phase"])
check("Phase 2 source record identity", m2["source_record"] == EXPECTED_SOURCE_RECORD, m2["source_record"])
check("Phase 2 source version identity", m2["source_version"] == EXPECTED_SOURCE_VERSION, m2["source_version"])
check("Phase 2 row/hash key agreement", set(m2["rows"]) == set(m2["export_sha256"]),
      f"rows={len(m2['rows'])}; hashes={len(m2['export_sha256'])}")

observed_phase1_hash = sha(manifest1_path)
check("Phase 1 manifest referenced by Phase 2", observed_phase1_hash == m2["phase1_manifest_sha256"],
      f"expected={m2['phase1_manifest_sha256']}; observed={observed_phase1_hash}")
m1 = json.loads(manifest1_path.read_text(encoding="utf-8-sig"))
check("Phase 1 source record identity", m1.get("source_record") == EXPECTED_SOURCE_RECORD, m1.get("source_record"))
check("Phase 1 raw-files-preserved status", bool(m1.get("raw_files_preserved")), m1.get("raw_files_preserved"))

STAGE_DIRS = {"Phase 1": p1_dir, "Phase 2": p2_dir, "Raw": raw_dir}
STAGE_HASHES = {"Phase 1": m1["export_sha256"], "Phase 2": m2["export_sha256"], "Raw": m1["input_sha256"]}
absent = [(STAGE_DIRS[s] / n).relative_to(project_root).as_posix()
          for s, hashes in STAGE_HASHES.items() for n in hashes if not (STAGE_DIRS[s] / n).is_file()]
if absent:
    raise FileNotFoundError("Required input missing: " + ", ".join(absent) +
                            ". Run scripts/download_data.py and Phases 01-02 in this project folder.")
phase1_paths = {n: p1_dir / n for n in m1["export_sha256"]}
phase2_paths = {n: p2_dir / n for n in m2["export_sha256"]}
raw_paths = {n: raw_dir / n for n in m1["input_sha256"]}

verified_path_hashes, verified_path_labels, verification_rows = {}, {}, []

def register_verified(stage, canonical_name, path, expected_hash, actual_hash, status, note):
    logical = path.relative_to(project_root).as_posix()
    verified_path_hashes[path] = actual_hash
    verified_path_labels[path] = logical
    verification_rows.append({"stage": stage, "logical_path": logical, "resolved_filename": path.name,
                              "expected_sha256": expected_hash, "actual_sha256": actual_hash,
                              "status": status, "resolution_note": note})

register_verified("Phase 2 manifest", EXPECTED_PHASE2_NAME, manifest2_path, "", observed_manifest_hash,
                  "RECORDED_NOT_PREPINNED", "standard repository path; manifest includes its run time, so it has no fixed upstream hash")
register_verified("Phase 1 manifest", EXPECTED_PHASE1_NAME, manifest1_path, m2["phase1_manifest_sha256"],
                  observed_phase1_hash, "PASS", "standard repository path; SHA-256 matched")
for stage, mapping, hashes in [("Phase 1", phase1_paths, m1["export_sha256"]),
                               ("Phase 2", phase2_paths, m2["export_sha256"]),
                               ("Raw", raw_paths, m1["input_sha256"])]:
    for name, expected in hashes.items():
        path = mapping[name]
        actual = sha(path)
        check("SHA-256: " + stage + "/" + name, actual == expected, f"file={path.name}; expected={expected}; observed={actual}")
        register_verified(stage, name, path, expected, actual, "PASS", "standard repository path; SHA-256 matched")

logical_hashes = {verified_path_labels[p]: d for p, d in verified_path_hashes.items()}
for stored_path, expected in m2["verified_input_sha256"].items():
    relative = stored_path.replace("\\", "/")
    check("Phase 2 input reference: " + relative, logical_hashes.get(relative) == expected,
          f"expected={expected}; observed={logical_hashes.get(relative, 'not verified')}")

t1 = {name: read_csv(phase1_paths[name]) for name in m1["export_sha256"]}
t2 = {name: read_csv(phase2_paths[name]) for name in m2["export_sha256"]}
check("Phase 1 export coverage", len(t1) == 11, len(t1))
check("Phase 2 export coverage", len(t2) == 26 and set(t2) == set(m2["rows"]), len(t2))
for name, expected in m2["rows"].items():
    check("Phase 2 row count: " + name, len(t2[name]) == int(expected), f"expected={expected}; observed={len(t2[name])}")

EXPECTED_PHASE2_CORE_COUNTS = {"worksheet_register.csv": 92, "measurement_points.csv": 4600,
                               "measurement_values_source.csv": 39100, "image_records_source.csv": 1031,
                               "validation_summary.csv": 43, "value_qc_findings.csv": 24}
for name, expected in EXPECTED_PHASE2_CORE_COUNTS.items():
    check("Phase 2 checkpoint invariant: " + name, len(t2[name]) == expected, f"expected={expected}; observed={len(t2[name])}")
check("Recorded Phase 2 checks passed", t2["validation_summary.csv"]["status"].eq("PASS").all(),
      t2["validation_summary.csv"]["status"].value_counts().to_dict())

input_verification = pd.DataFrame(verification_rows)

EXPECTED_SCHEMAS = {'entity_schema.csv': ['table', 'row_granularity', 'primary_key', 'parent_scope', 'scientific_boundary'], 'source_header_inventory.csv': ['source_header', 'occurrences', 'blank_cells'], 'image_index_column_profile.csv': ['column', 'rows', 'blank_count', 'blank_percent', 'distinct_nonblank'], 'value_qc_findings.csv': ['rule', 'severity', 'source', 'location', 'evidence', 'action'], 'phase2_issue_decisions.csv': ['phase1_issue_id', 'issue_code', 'phase2_status', 'decision'], 'audit_issue_log.csv': ['issue_id', 'issue_code', 'severity', 'source', 'sheet', 'column', 'evidence', 'next_action'], 'validation_summary.csv': ['check', 'status', 'evidence'], 'missing_value_rules.csv': ['source_condition', 'interpretation', 'action'], 'image_missingness.csv': ['source_column', 'rows', 'unknown_blank'], 'phase1_issue_carry_forward.csv': ['issue_id', 'issue_code', 'severity', 'source', 'sheet', 'column', 'evidence', 'next_action'], 'strain_conversion_preview.csv': ['member_path', 'sheet', 'source_excel_row', 'source_column_position', 'source_header', 'source_value_json', 'source_unit', 'numeric_value', 'parse_status', 'conversion_factor', 'standardised_value', 'standardised_unit', 'conversion_rule', 'transformation_status'], 'measurement_points.csv': ['member_path', 'sheet', 'source_excel_row'], 'worksheet_register.csv': ['member_path', 'sheet', 'source_stem', 'member_sha256', 'measurement_family', 'header_row', 'point_count', 'physical_sample_link_status'], 'context_review.csv': ['question', 'observed', 'interpretation'], 'measurement_values_source.csv': ['member_path', 'sheet', 'source_excel_row', 'source_column_position', 'source_header', 'source_value_json', 'source_unit', 'numeric_value', 'parse_status'], 'image_field_dictionary.csv': ['source_column', 'logical_type', 'definition', 'semantic_limit'], 'data_dictionary.csv': ['table', 'field', 'logical_type', 'definition', 'provenance', 'unit_policy', 'missingness_policy'], 'image_filename_expectations.csv': ['source_csv_record', 'image_id', 'expected_filename', 'existence_status'], 'source_manifest.csv': ['source_record', 'source_version', 'metadata_review_date', 'canonical_filename', 'selected_filename', 'relative_path', 'bytes', 'md5', 'reference_md5', 'checksum_match', 'sha256', 'dataset_licence_status'], 'measurement_context_summary.csv': ['measurement_family', 'source_header', 'worksheets', 'minimum', 'maximum', 'fewest_distinct_values', 'most_distinct_values'], 'quality_control_rules.csv': ['rule', 'severity', 'criterion', 'action'], 'deferred_entities.csv': ['entity', 'reason_not_instantiated'], 'workbook_structure_inventory.csv': ['member_path', 'sheet', 'reported_max_row', 'reported_max_column', 'candidate_header_row', 'data_rows', 'blank_data_rows', 'duplicate_rows', 'formula_cells', 'excel_error_cells', 'headers_json'], 'image_index_value_counts.csv': ['column', 'source_value', 'is_blank', 'count'], 'input_verification.csv': ['relative_path', 'sha256', 'status'], 'measurement_column_profile.csv': ['member_path', 'sheet', 'column_position', 'excel_column', 'source_header', 'observed_rows', 'blank_count', 'blank_percent', 'numeric_count', 'nonnumeric_nonblank_count', 'nonfinite_numeric_count', 'distinct_nonblank', 'numeric_min', 'numeric_max', 'numeric_median'], 'archive_member_inventory.csv': ['member_path', 'is_directory', 'folder', 'suffix', 'bytes', 'compressed_bytes', 'crc32', 'sha256'], 'additional_review_items.csv': ['review_topic', 'status', 'next_action'], 'image_observed_categories.csv': ['column', 'source_value', 'is_blank', 'count'], 'literature_evidence_register.csv': ['document', 'source_location', 'source_statement', 'scope', 'schema_decision'], 'measurement_axis_profile.csv': ['member_path', 'sheet', 'source_header', 'numeric_min', 'numeric_max', 'distinct_numeric_values'], 'linkage_summary.csv': ['quantity', 'count'], 'filename_candidates_unconfirmed.csv': ['source_stem', 'GG_count', 'viscosity_count', 'GG_members', 'viscosity_members', 'exact_one_each', 'physical_sample_link'], 'candidate_filename_correspondences.csv': ['source_stem', 'GG_count', 'viscosity_count', 'GG_members', 'viscosity_members', 'exact_one_each', 'physical_sample_link'], 'source_to_schema_mapping.csv': ['source_header', 'canonical_field', 'logical_type', 'source_unit', 'conversion_policy', 'definition'], 'reviewed_inputs.csv': ['reviewed_filename', 'sha256', 'role'], 'image_records_source.csv': ['source_csv_record', 'image_id', 'date', 'ink_type_ph', 'shape', 'intensity', 'completeness_sam', 'contain_sps', 'image_id_status', 'date_status', 'ink_type_ph_status', 'shape_status', 'intensity_status', 'completeness_sam_status', 'contain_sps_status', 'expected_filename', 'existence_status']}

for tables in [t1, t2]:
    for filename, table in tables.items():
        expected = EXPECTED_SCHEMAS[filename]
        check("Schema: " + filename, list(table.columns) == expected, f"expected={expected}; observed={list(table.columns)}")

phase_check("P03-03", "All inputs match their manifests, counts and schemas", True,
       f"{len(verification_rows)} files verified; {len(checks)} handoff checks passed")
differs = [n for n, h in PHASE02_REFERENCE_SHA256.items() if m2["export_sha256"].get(n) != h]
phase_check("P03-04", "Phase 02 tables identical to the reference run", not differs,
       "26 of 26 tables identical" if not differs else "differ: " + ", ".join(differs) + " — review the changed Phase 02 output first")

[PASS] P03-03: 41 files verified; 133 handoff checks passed
[PASS] P03-04: 26 of 26 tables identical


## 3. Evidence and transformation decisions

The evidence register records where each decision comes from. The publication and supplementary document are listed with the checksums of the copies reviewed when the rules were written; they are reference records only and are not needed to run the notebook. The decision table states, per topic, whether a transformation is supported, deferred or out of scope.

*Scientific note:* the paper reports viscosity in Pa·s, but that does not define the raw export token `mPas`, so viscosity conversion stays deferred.

In [3]:
evidence_register = pd.DataFrame([
 {'evidence_id':'DOC_METHOD','source':'Main publication, DOI 10.1080/17452759.2026.2671497',
  'locator':'Section 5.1; PDF page 15 / printed page 14',
  'sha256':'c3868eb63b4f4ce96be0b39ea4b058cc91ccf86d98697ff4ca673bfb560b6536',
  'observation':'HAAKE MARS III; nominal shear-rate range 0.01–1000 s^-1; amplitude sweeps at 1 Hz and 22.5 °C.',
  'boundary':'Method context does not replace recorded conditions or define the export token mPas.'},
 {'evidence_id':'DOC_SUPPLEMENT','source':'Supplied supplementary document (reviewed copy)',
  'locator':'Supplementary text and Figure S2 caption',
  'sha256':'16bfe75b0ddf0be5029d384fed2f888ffdad9c0fe80e2803850d0b1cd2f98e81',
  'observation':'Supplementary captions describe fitted parameters and model evaluation; no export-token definition was identified in the reviewed text.',
  'boundary':'A viscosity label in a fitted-parameter figure would not alone establish raw export notation.'},
 {'evidence_id':'SOURCE_HEADERS','source':'Pinned raw rheology ZIP and Phase 2 source mapping',
  'locator':'Original worksheet header cells; checked in Section 4', 'sha256':m1['input_sha256']['ALG-Ph_HA-Ph_rheology_data.zip'],
  'observation':'Exact strain headers distinguish percent from source-labelled fraction.',
  'boundary':'The conversion respects labels; it cannot establish that the instrument labels were scientifically correct.'}
])
decisions = pd.DataFrame([
 ['STRAIN','supported_by_header','Divide explicitly labelled percent by 100; retain labelled fractions.','SOURCE_HEADERS'],
 ['VISCOSITY','unresolved','Retain mPas and source numeric values; standardised viscosity value/unit remain blank.','DOC_METHOD;DOC_SUPPLEMENT'],
 ['IMAGE_ANNOTATIONS','unresolved','Retain literal annotations and unknown blanks; do not infer outcomes or reagent presence.','Phase 2 image dictionary'],
 ['INTENSITY','unresolved','Retain the source token; do not convert to physical irradiance.','Phase 2 image dictionary'],
 ['ACQUISITION_CONTEXT','source_preserved','Keep recorded conditions attached through exact point coordinates.','SOURCE_HEADERS;DOC_METHOD'],
 ['PHYSICAL_LINKAGE','deferred_phase4','No physical samples, formulations or replicate IDs are assigned.','Phase 2 linkage summary'],
 ['LVR_AND_FITTING','out_of_scope','Do not choose an LVR, fit curves or derive representative material properties.','DOC_METHOD']
],columns=['topic','decision_status','decision','evidence_reference'])
decisions

,topic,decision_status,decision,evidence_reference
0,STRAIN,supported_by_header,Divide explicitly labelled percent by 100; retain labelled fractions.,SOURCE_HEADERS
1,VISCOSITY,unresolved,Retain mPas and source numeric values; standardised viscosity value/unit remain blank.,DOC_METHOD;DOC_SUPPLEMENT
2,IMAGE_ANNOTATIONS,unresolved,Retain literal annotations and unknown blanks; do not infer outcomes or reagent presence.,Phase 2 image dictionary
3,INTENSITY,unresolved,Retain the source token; do not convert to physical irradiance.,Phase 2 image dictionary
4,ACQUISITION_CONTEXT,source_preserved,Keep recorded conditions attached through exact point coordinates.,SOURCE_HEADERS;DOC_METHOD
5,PHYSICAL_LINKAGE,deferred_phase4,"No physical samples, formulations or replicate IDs are assigned.",Phase 2 linkage summary
6,LVR_AND_FITTING,out_of_scope,"Do not choose an LVR, fit curves or derive representative material properties.",DOC_METHOD


## 4. Reconcile every source cell with the original workbooks

Keys are checked for uniqueness and parent links. Each workbook is then re-read from the archive, and every one of the 39,100 Phase 02 source cells is compared with the original cell: same position, header, unit and value. Image-index records are compared field by field with the original CSV. Any difference stops the notebook.

*Scientific note:* the points along one curve are related observations of one acquisition. Treating them as independent replicates (pseudoreplication) would inflate any later statistics; the keys kept here allow grouping by worksheet later.

In [4]:
values = t2['measurement_values_source.csv'].copy()
points = t2['measurement_points.csv'].copy()
worksheets = t2['worksheet_register.csv'].copy()
images = t2['image_records_source.csv'].copy()
mapping = t2['source_to_schema_mapping.csv'].copy()
WK = ['member_path','sheet']
PK = WK + ['source_excel_row']
CK = PK + ['source_column_position']
for label,table,key in [('worksheet',worksheets,WK),('point',points,PK),('cell',values,CK),
                        ('image record',images,['source_csv_record'])]:
    check(label+' keys nonblank',table[key].ne('').all().all())
    check(label+' keys unique',not table.duplicated(key).any())
check('Points have worksheet parents',len(points.merge(worksheets[WK],on=WK,how='inner',validate='many_to_one'))==len(points))
check('Cells have point parents',len(values.merge(points[PK],on=PK,how='inner',validate='many_to_one'))==len(values))
check('Header mapping unique',not mapping.source_header.duplicated().any())
check('Header mapping coverage',set(values.source_header)==set(mapping.source_header))
lookup = values.set_index(CK)
source_headers = mapping.set_index('source_header').source_unit.to_dict()
verified_cells = 0
with zipfile.ZipFile(raw_paths['ALG-Ph_HA-Ph_rheology_data.zip']) as archive:
    check('Raw workbook coverage',set(n for n in archive.namelist() if n.lower().endswith('.xlsx'))==set(worksheets.member_path))
    for member,group in worksheets.groupby('member_path',sort=False):
        payload = archive.read(member)
        check('Member hash: '+member,group.member_sha256.eq(hashlib.sha256(payload).hexdigest()).all())
        wb = openpyxl.load_workbook(io.BytesIO(payload),read_only=True,data_only=False)
        try:
            check('Sheet coverage: '+member,set(wb.sheetnames)==set(group.sheet))
            for w in group.itertuples(index=False):
                rows = list(wb[w.sheet].iter_rows())
                check('Header row: '+member,int(w.header_row)==1)
                headers = ['' if c.value is None else str(c.value) for c in rows[0]]
                active = [(i,row) for i,row in enumerate(rows[1:],2)
                          if not all(c.value is None or (isinstance(c.value,str) and not c.value.strip()) for c in row)]
                check('Raw point coverage: '+member,len(active)==int(w.point_count))
                for row_number,row in active:
                    for col,(header,cell) in enumerate(zip(headers,row),1):
                        key = (member,w.sheet,str(row_number),str(col))
                        if key not in lookup.index:
                            raise ValueError('Unrepresented source cell: '+str(key))
                        r = lookup.loc[key]
                        if cell.data_type in {'f','e'} or r.source_header != header or r.source_unit != source_headers[header]:
                            raise ValueError('Source header/type mismatch: '+str(key))
                        if r.source_value_json != json.dumps(cell.value,ensure_ascii=False,allow_nan=False):
                            raise ValueError('Source scalar mismatch: '+str(key))
                        verified_cells += 1
        finally:
            wb.close()
check('All raw cells reconciled',verified_cells==len(values),verified_cells)
with raw_paths['image_index.csv'].open(encoding='utf-8-sig',newline='') as f:
    reader = csv.DictReader(f); raw_image_fields = reader.fieldnames; raw_images = list(reader)
check('Raw image count',len(raw_images)==len(images))
check('Image dictionary coverage',set(raw_image_fields)==set(t2['image_field_dictionary.csv'].source_column))
for i,raw in enumerate(raw_images,2):
    record = images.loc[images.source_csv_record.eq(str(i))]
    if len(record)!=1 or any(record.iloc[0][field]!=raw[field] for field in raw_image_fields):
        raise ValueError('Image source mismatch at CSV record '+str(i))
phase_check('P03-05', 'Every source cell and image record matches the original files',
       verified_cells == EXPECTED['cells'] and len(images) == EXPECTED['images'],
       f'{verified_cells:,} cells and {len(images):,} image records reconciled')

[PASS] P03-05: 39,100 cells and 1,031 image records reconciled


## 5. Apply the transformation rules

One rule is created per source header from the Phase 02 mapping. Each value is re-parsed from its preserved source scalar and must agree with Phase 02. Only `identity` and `divide by 100` rules produce a standardised value; deferred viscosity rows keep their source number with an empty standardised value and the status `deferred_unit_semantics`. All original columns are kept; new columns are added beside them.

In [5]:
rule_rows = []
for i,r in enumerate(mapping.itertuples(index=False),1):
    policy = r.conversion_policy
    check('Known conversion policy: '+r.source_header,policy in {'retain','identity','divide by 100','deferred'})
    target = ('1 (strain fraction)' if r.canonical_field=='strain_fraction' else r.source_unit) if policy in {'identity','divide by 100'} else ''
    rule_rows.append({'rule_id':f'TR{i:02d}','source_header':r.source_header,'canonical_field':r.canonical_field,
        'source_unit':r.source_unit,'operation':policy,'factor':0.01 if policy=='divide by 100' else 1.0 if policy=='identity' else np.nan,
        'target_unit':target,'evidence':'Exact source header; Phase 2 mapping',
        'scientific_caution':('Retained as labelled fraction; unusual values are not reinterpreted as percent.' if r.source_header=='ɣ in -' else 'Unit token unresolved; no Pa·s value approved.' if policy=='deferred' else 'Representation only; no physical-sample inference.'),
        'approval_status':'deferred' if policy=='deferred' else 'text_retention' if policy=='retain' else 'supported_by_header'})
rules = pd.DataFrame(rule_rows)
standardised = values.merge(rules.drop(columns='source_unit'),on='source_header',how='left',validate='many_to_one',sort=False)
parsed = []
for r in standardised.itertuples(index=False):
    scalar = json.loads(r.source_value_json)
    if r.operation=='retain':
        parsed.append(np.nan); continue
    if scalar is None or (isinstance(scalar,str) and not scalar.strip()):
        status,num='blank_unknown',np.nan
    elif isinstance(scalar,bool):
        status,num='invalid_boolean',np.nan
    else:
        try:
            num=float(scalar); status='numeric' if np.isfinite(num) else 'nonfinite'
        except (TypeError,ValueError):
            status,num='nonnumeric',np.nan
    if status!=r.parse_status:
        raise ValueError('Parse status differs from checkpoint: '+str((r.member_path,r.sheet,r.source_excel_row,r.source_column_position)))
    if status=='numeric' and not np.isclose(num,float(r.numeric_value),rtol=1e-14,atol=0):
        raise ValueError('Parsed scalar differs from checkpoint numeric value.')
    parsed.append(num)
standardised['source_numeric_checked']=parsed
eligible=standardised.operation.isin(['identity','divide by 100']) & standardised.parse_status.eq('numeric')
standardised['standardised_value']=np.nan
standardised.loc[eligible,'standardised_value']=standardised.loc[eligible,'source_numeric_checked']*standardised.loc[eligible,'factor']
standardised['standardised_unit']=standardised.target_unit
standardised['transformation_status']=np.select(
    [standardised.operation.eq('deferred'),standardised.operation.eq('retain'),eligible],
    ['deferred_unit_semantics','text_preserved','applied'],default='not_applied_parse_review')
standardised=standardised.merge(worksheets[WK+['measurement_family']],on=WK,validate='many_to_one',sort=False)
check('Cell count preserved after transformation',len(standardised)==len(values))
check('Original columns preserved',standardised[values.columns].equals(values))
check('No viscosity conversion',standardised.loc[standardised.operation.eq('deferred'),'standardised_value'].isna().all())
check('Finite standardised values',np.isfinite(standardised.loc[eligible,'standardised_value']).all())
status_counts = standardised.transformation_status.value_counts().to_dict()
phase_check('P03-06', 'Transformations applied only where supported', len(standardised) == EXPECTED['cells'],
       '; '.join(f'{k} {v:,}' for k, v in status_counts.items()))
standardised.groupby(['canonical_field','transformation_status'],sort=False).size().rename('source_cells').reset_index()

[PASS] P03-06: applied 29,900; text_preserved 4,600; deferred_unit_semantics 4,600


,canonical_field,transformation_status,source_cells
0,segment_index_source,text_preserved,4600
1,time_s,applied,4600
2,segment_time_s,applied,4600
3,shear_stress_pa,applied,4600
4,storage_modulus_pa,applied,2300
5,loss_modulus_pa,applied,2300
6,complex_viscosity_source,deferred_unit_semantics,2300
7,strain_fraction,applied,2300
8,frequency_hz,applied,2300
9,temperature_c,applied,4600


## 6. Check the strain conversion

Every strain cell is matched to the Phase 02 preview by its source key; the standardised values and factors must agree, and reversing the conversion must return the source value. There are 100 percentage-labelled and 2,200 fraction-labelled strain cells.

*Scientific note:* a fraction of 0.01 equals 1 % strain. Large values labelled as fractions are kept as labelled, not reinterpreted as percentages.

In [6]:
strain = standardised.loc[standardised.canonical_field.eq('strain_fraction')].copy()
preview = t2['strain_conversion_preview.csv']
check('Strain preview key uniqueness',not preview.duplicated(CK).any())
joined = strain.merge(preview[CK+['standardised_value','conversion_factor']],on=CK,how='outer',
                      suffixes=('','_preview'),indicator=True,validate='one_to_one')
check('Strain preview coverage',joined['_merge'].eq('both').all())
check('Strain arithmetic agrees with preview',np.allclose(joined.standardised_value.astype(float),joined.standardised_value_preview.astype(float),rtol=1e-12,atol=1e-15))
check('Strain factors agree with preview',np.allclose(joined.factor.astype(float),joined.conversion_factor.astype(float),rtol=0,atol=0))
check('Percent strain count',strain.source_header.eq('ɣ in %').sum()==100)
check('Fraction strain count',strain.source_header.eq('ɣ in -').sum()==2200)
check('Strain round trip',np.allclose(strain.standardised_value/strain.factor,strain.source_numeric_checked,rtol=1e-12,atol=1e-15))
phase_check('P03-07', 'Strain conversion matches the preview and reverses exactly', True,
       f"{int(strain.source_header.eq('ɣ in %').sum())} percent cells divided by 100; {int(strain.source_header.eq('ɣ in -').sum()):,} fraction cells kept")

[PASS] P03-07: 100 percent cells divided by 100; 2,200 fraction cells kept


## 7. Image records: keep blanks unknown and standardise dates

Blank statuses from Phase 02 are re-checked against the values. A valid `YYYY/MM/DD` date gains an ISO copy (`YYYY-MM-DD`), checked by converting it back. No annotation is converted to yes/no, and intensity stays a source token.

In [7]:
image_standardised=images.copy()
for field in raw_image_fields:
    blank=images[field].str.strip().eq('')
    check('Image blank status: '+field,images[field+'_status'].eq('unknown_blank').equals(blank))
shape_ok=images.date.str.fullmatch(r'\d{4}/\d{2}/\d{2}')
parsed_dates=pd.to_datetime(images.date.where(shape_ok),format='%Y/%m/%d',errors='coerce')
image_standardised['date_iso']=parsed_dates.dt.strftime('%Y-%m-%d').fillna('')
image_standardised['date_transformation_status']=np.where(shape_ok & parsed_dates.notna(),'format_standardised','unresolved_date')
check('Image original columns preserved',image_standardised[images.columns].equals(images))

valid_dates = shape_ok & parsed_dates.notna()
check('Date round trip', parsed_dates.loc[valid_dates].dt.strftime('%Y/%m/%d').equals(images.loc[valid_dates,'date']))
image_standardised['date_conversion_rule']='DATE01: strict YYYY/MM/DD to YYYY-MM-DD; no event semantics inferred'
n_dates = int(image_standardised.date_transformation_status.eq('format_standardised').sum())
phase_check('P03-08', 'Image blanks unchanged and dates standardised', n_dates == EXPECTED['images'],
       f'{n_dates:,} of {len(images):,} dates standardised; ' + '; '.join(f"{f} {int(images[f+'_status'].eq('unknown_blank').sum())} unknown" for f in raw_image_fields if images[f+'_status'].eq('unknown_blank').any()))

[PASS] P03-08: 1,031 of 1,031 dates standardised; completeness_sam 943 unknown; contain_sps 922 unknown


## 8. Link QC findings to the cells they concern

Each of the 24 Phase 02 findings gets an ID (`P2-QC-001` …) and is linked to its exact source cell, or for a column-level image finding, to every blank record. Non-positive responses and acquisition axes are recomputed and must land on the same cells. One new review finding, `P3-QC-001`, links every shear-rate cell below the paper's stated 0.01 s⁻¹. A link count is a count of affected records, not of samples.

In [8]:
prior=t2['value_qc_findings.csv'].copy()
prior.insert(0,'finding_id',[f'P2-QC-{i:03d}' for i in range(1,len(prior)+1)])
prior['finding_origin']='Phase 2 value_qc_findings.csv; one-based data-record order'
links=[]
for r in prior.itertuples(index=False):
    if r.rule=='IMAGE_BLANK':
        check('Image finding field',r.location in raw_image_fields,r.location)
        matches=images.loc[images[r.location].str.strip().eq('')]
        for im in matches.itertuples(index=False):
            links.append({'finding_id':r.finding_id,'target_type':'image_field','member_path':'','sheet':'',
              'source_excel_row':'','source_column_position':'','source_csv_record':im.source_csv_record,'image_field':r.location})
    else:
        match=re.fullmatch(r'(.*); row (\d+); column (\d+); (.*)',r.location)
        check('Parse exact QC source location: '+r.finding_id,match is not None)
        sheet,row,col,header=match.groups()
        hit=values.loc[values.member_path.eq(r.source)&values.sheet.eq(sheet)&values.source_excel_row.eq(row)&values.source_column_position.eq(col)]
        check('QC location resolves: '+r.finding_id,len(hit)==1 and hit.iloc[0].source_header==header)
        links.append({'finding_id':r.finding_id,'target_type':'measurement_cell','member_path':r.source,'sheet':sheet,
                      'source_excel_row':row,'source_column_position':col,'source_csv_record':'','image_field':''})
qc_links=pd.DataFrame(links)
check('All Phase 2 findings linked',set(qc_links.finding_id)==set(prior.finding_id))
num=standardised.source_numeric_checked
response=standardised.source_header.isin(["G' in Pa",'G" in Pa','η in mPas','|η*| in mPas'])
axis_bad=(standardised.source_header.isin(['f in Hz','ɣ̇ in 1/s']) & num.le(0)) | (standardised.canonical_field.eq('strain_fraction') & num.lt(0))
for rule,mask in [('NONPOSITIVE_RESPONSE',response & num.le(0)),('ACQUISITION_AXIS',axis_bad)]:
    expected_ids=prior.loc[prior.rule.eq(rule),'finding_id']
    expected=qc_links.loc[qc_links.finding_id.isin(expected_ids),CK]
    actual=standardised.loc[mask,CK]
    check('Recomputed locations: '+rule,set(map(tuple,expected.to_numpy()))==set(map(tuple,actual.to_numpy())),len(actual))
# A document-range difference is a review flag, not an exclusion rule.
below=standardised.loc[standardised.source_header.eq('ɣ̇ in 1/s') & num.lt(0.01)]
new_findings=pd.DataFrame([{'finding_id':'P3-QC-001','rule':'BELOW_NOMINAL_METHOD_RANGE','severity':'REVIEW',
    'source':'Main publication Section 5.1 versus observed source cells','location':'shear-rate cells < 0.01 1/s',
    'evidence':f'{len(below)} cells; nominal lower bound is 0.01 1/s',
    'action':'Retain all observations; do not impose the publication range as a filter.',
    'finding_origin':'Phase 3 document-context review'}])
new_links=[{'finding_id':'P3-QC-001','target_type':'measurement_cell',**dict(zip(CK,k)),
            'source_csv_record':'','image_field':''} for k in below[CK].itertuples(index=False,name=None)]
qc_links=pd.concat([qc_links,pd.DataFrame(new_links)],ignore_index=True)
qc_findings=pd.concat([prior,new_findings],ignore_index=True)
check('QC association uniqueness',not qc_links.duplicated().any())
cell_ids=qc_links.loc[qc_links.target_type.eq('measurement_cell')].groupby(CK,sort=False).finding_id.agg(lambda x:';'.join(sorted(set(x)))).rename('qc_finding_ids').reset_index()
standardised=standardised.merge(cell_ids,on=CK,how='left',validate='one_to_one',sort=False)
standardised['qc_finding_ids']=standardised.qc_finding_ids.fillna('')
standardised['qc_status']=np.where(standardised.qc_finding_ids.ne(''),'flagged_for_review','no_flag_from_applied_value_checks')
image_ids=qc_links.loc[qc_links.target_type.eq('image_field')].groupby('source_csv_record').finding_id.agg(lambda x:';'.join(sorted(set(x)))).rename('qc_finding_ids').reset_index()
image_standardised=image_standardised.merge(image_ids,on='source_csv_record',how='left',validate='one_to_one',sort=False)
image_standardised['qc_finding_ids']=image_standardised.qc_finding_ids.fillna('')
check('QC additions preserve source cells',standardised[values.columns].equals(values))
check('QC additions preserve image records',image_standardised[images.columns].equals(images))
phase_check('P03-09', 'QC findings linked to exact cells and records', len(qc_findings) == EXPECTED['qc_findings'],
       f'{len(qc_findings)} findings (24 from Phase 02, 1 new); {len(qc_links):,} cell/record links; {len(below)} shear-rate cells below 0.01 1/s')
qc_findings.groupby(['finding_origin','rule']).size().rename('finding_records').reset_index()

[PASS] P03-09: 25 findings (24 from Phase 02, 1 new); 2,302 cell/record links; 415 shear-rate cells below 0.01 1/s


,finding_origin,rule,finding_records
0,Phase 2 value_qc_findings.csv; one-based data-record order,ACQUISITION_AXIS,2
1,Phase 2 value_qc_findings.csv; one-based data-record order,IMAGE_BLANK,2
2,Phase 2 value_qc_findings.csv; one-based data-record order,NONPOSITIVE_RESPONSE,20
3,Phase 3 document-context review,BELOW_NOMINAL_METHOD_RANGE,1


## 9. Point view, final checks and outputs

A convenience view with one row per measurement point keeps every condition (frequency, strain, temperature, shear rate, time) beside its responses; the long cell table remains the authoritative record. Final checks confirm coverage, no duplicate rows, and the retained non-positive values. Before writing, every input is re-hashed. The 13 tables and the run manifest replace any earlier run in `data/processed/standardisation_and_quality_checks/`.

In [9]:
check('One canonical field per source point',not standardised.duplicated(PK+['canonical_field']).any())
view=standardised[PK+['canonical_field']].copy()
view['view_value']=[r.standardised_value if r.transformation_status=='applied' else
                    json.loads(r.source_value_json) for r in standardised.itertuples(index=False)]
point_view=view.pivot(index=PK,columns='canonical_field',values='view_value').reset_index()
point_view.columns.name=None
point_view=points.merge(point_view,on=PK,how='left',validate='one_to_one',sort=False)
point_view=point_view.merge(worksheets[WK+['measurement_family','physical_sample_link_status']],on=WK,validate='many_to_one',sort=False)
point_view['viscosity_unit_status']='mPas retained; conversion unresolved'
point_view['observation_unit']='source measurement point; physical sample unconfirmed'
check('Point-view coverage',len(point_view)==len(points)==EXPECTED_PHASE2_CORE_COUNTS['measurement_points.csv'],len(point_view))
check('Source-cell coverage',len(standardised)==EXPECTED_PHASE2_CORE_COUNTS['measurement_values_source.csv'],len(standardised))
check('Image coverage',len(image_standardised)==EXPECTED_PHASE2_CORE_COUNTS['image_records_source.csv'],len(image_standardised))
check('Physical links remain unresolved',point_view.physical_sample_link_status.eq('unresolved').all())
for field in ['frequency_hz','strain_fraction','temperature_c','shear_rate_per_s','time_s','segment_time_s','shear_stress_pa']:
    expected=int(standardised.canonical_field.eq(field).sum())
    actual=int(point_view[field].notna().sum())
    check('Point-view context coverage: '+field,actual==expected,actual)
# Duplicate source rows are reviewed within a worksheet; no rows are dropped.
row_signatures=values.groupby(PK,sort=False).source_value_json.agg(tuple).reset_index(name='signature')
duplicate_points=row_signatures.loc[row_signatures.duplicated(WK+['signature'],keep=False)].drop(columns='signature')
check('No unexplained duplicate full rows in pinned snapshot',len(duplicate_points)==0,len(duplicate_points))
check('Dates valid in pinned snapshot',image_standardised.date_transformation_status.eq('format_standardised').all())
check('Retained negative response cells',int((response & num.le(0)).sum())==20)
check('Retained nonpositive acquisition-axis cells',int(axis_bad.sum())==2)
check('Image blank counts retained',images.completeness_sam.eq('').sum()==943 and images.contain_sps.eq('').sum()==922)
context_coverage=standardised.groupby(['measurement_family','canonical_field','source_unit','standardised_unit'],dropna=False).size().rename('source_cells').reset_index()
context_coverage

,measurement_family,canonical_field,source_unit,standardised_unit,source_cells
0,oscillatory_moduli,complex_viscosity_source,mPas,,2300
1,oscillatory_moduli,frequency_hz,Hz,Hz,2300
2,oscillatory_moduli,loss_modulus_pa,Pa,Pa,2300
3,oscillatory_moduli,segment_index_source,not applicable,,2300
4,oscillatory_moduli,segment_time_s,s,s,2300
5,oscillatory_moduli,shear_stress_pa,Pa,Pa,2300
6,oscillatory_moduli,storage_modulus_pa,Pa,Pa,2300
7,oscillatory_moduli,strain_fraction,%,1 (strain fraction),100
8,oscillatory_moduli,strain_fraction,-,1 (strain fraction),2200
9,oscillatory_moduli,temperature_c,°C,°C,2300


In [10]:
required_ids = {f'P03-{i:02d}' for i in range(1, 10)}
passed_ids = {c['check_id'] for c in phase_checks if c['status'] == 'PASS'}
if not required_ids <= passed_ids:
    raise RuntimeError('Outputs not written: checks not yet passed in this run: ' + ', '.join(sorted(required_ids - passed_ids)))

# Phase 02 history tables stay at their verified paths and are not duplicated here.
unresolved = t2['additional_review_items.csv'].copy()
unresolved['phase3_disposition'] = [
    'Conversion remains deferred; see evidence and decision registers.' if topic == 'VISCOSITY_UNIT_TOKEN' else
    'Retained as an open question; no LVR selection, population inference or forced exclusion.'
    for topic in unresolved.review_topic
]

# Re-hash every protected input immediately before export.
for path, original in verified_path_hashes.items():
    logical = verified_path_labels[path]
    check('Input preserved: ' + logical, sha(path) == original, path.name)

issue_decisions = t2['phase2_issue_decisions.csv'].copy()
issue_decisions['phase3_decision'] = [
    'Source-label strain standardisation implemented and validated; unusual fraction semantics remain open.'
    if code == 'MIXED_STRAIN_UNITS' else
    'Prior decision retained; no new scientific interpretation or forced closure.'
    for code in issue_decisions.issue_code
]
check(
    'Phase 1 issue history preserved',
    len(issue_decisions) == 15 and
    set(issue_decisions.phase1_issue_id) == set(t2['phase1_issue_carry_forward.csv'].issue_id)
)

exports = {
    'input_verification.csv': input_verification,
    'document_evidence_register.csv': evidence_register,
    'standardisation_decisions.csv': decisions,
    'phase3_issue_decisions.csv': issue_decisions,
    'transformation_rules.csv': rules,
    'measurement_values_standardised.csv': standardised,
    'measurement_points_view.csv': point_view,
    'image_records_standardised.csv': image_standardised,
    'qc_findings.csv': qc_findings,
    'qc_observation_links.csv': qc_links,
    'acquisition_context_coverage.csv': context_coverage,
    'unresolved_review_items.csv': unresolved,
}

OUT.mkdir(parents=True, exist_ok=True)
for name, table in exports.items():
    target = OUT / name
    temporary = target.with_suffix('.csv.tmp')
    table.to_csv(temporary, index=False, encoding='utf-8-sig', lineterminator='\n')
    check('Export row count: ' + name, len(read_csv(temporary)) == len(table), len(table))
    temporary.replace(target)

check(
    'Protected inputs unchanged after export',
    all(sha(path) == digest for path, digest in verified_path_hashes.items())
)

validation_summary = pd.DataFrame(checks)
exports['validation_summary.csv'] = validation_summary
validation_summary.to_csv(
    OUT / 'validation_summary.csv',
    index=False,
    encoding='utf-8-sig',
    lineterminator='\n'
)

input_sha256_logical = {
    verified_path_labels[path]: digest
    for path, digest in verified_path_hashes.items()
}
input_resolution = {
    verified_path_labels[path]: {
        'resolved_filename': path.name,
        'resolution_note': next(
            (row['resolution_note'] for row in verification_rows
             if row['logical_path'] == verified_path_labels[path]),
            ''
        )
    }
    for path in verified_path_hashes
}

manifest = {
    'phase': '3: standardisation and quality checks',
    'notebook': '03_standardisation_and_quality_checks.ipynb',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'source_record': m2['source_record'],
    'source_version': m2['source_version'],
    'phase2_manifest_sha256': sha(manifest2_path),
    'phase1_manifest_sha256': m2['phase1_manifest_sha256'],
    'input_sha256': input_sha256_logical,
    'input_resolution': input_resolution,
    'environment': runtime.to_dict('records'),
    'input_discovery_mode': 'standard repository layout',
    'phase2_manifest_discovery': 'standard repository path',
    'output_location_mode': 'standard repository path: data/processed/standardisation_and_quality_checks',
    'phase2_manifest_hash_status': 'recorded at Phase 3 runtime; no independent upstream self-hash was available',
    'rows': {name: len(table) for name, table in exports.items()},
    'export_sha256': {name: sha(OUT / name) for name in exports},
    'input_rows': {name: len(table) for name, table in t2.items()},
    'transformation_status_counts': {
        str(k): int(v) for k, v in standardised.transformation_status.value_counts().items()
    },
    'qc_rule_counts': {str(k): int(v) for k, v in qc_findings.rule.value_counts().items()},
    'qc_associations': len(qc_links),
    'validation_counts': {
        str(k): int(v) for k, v in validation_summary.status.value_counts().items()
    },
    'unresolved_decisions': decisions.loc[
        decisions.decision_status.isin(['unresolved', 'deferred_phase4'])
    ].to_dict('records'),
    'rules_version': 'phase3-v3-standard-repository-layout',
    'physical_sample_linkage': 'unresolved; Phase 4 deferred',
    'viscosity_conversion': 'deferred; original mPas retained',
    'strain_conversion': 'explicit source percent divided by 100; source fraction retained',
    'image_blanks': 'unknown; preserved',
    'raw_and_checkpoints_preserved': True,
    'dataset_redistribution': 'unresolved; source-derived exports for local review',
    'model_readiness': 'not established'
}

target = OUT / 'standardisation_run_manifest.json'
temporary = target.with_suffix('.json.tmp')
temporary.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False),
    encoding='utf-8'
)
temporary.replace(target)

n_ledger_pass = int(validation_summary.status.eq('PASS').sum())
phase_check('P03-10', 'Structural ledger', n_ledger_pass == len(validation_summary) == EXPECTED['ledger_checks'],
       f'{n_ledger_pass} of {len(validation_summary)} checks passed (exported as validation_summary.csv)')
phase_check('P03-11', 'Inputs unchanged and outputs written', len(exports) == EXPECTED['exports'],
       f'{len(verified_path_hashes)} inputs unchanged; {len(exports)} tables and standardisation_run_manifest.json written to {OUT_REL.as_posix()}/')
comparable = {n: h for n, h in manifest['export_sha256'].items() if n not in RUN_SPECIFIC_TABLES}
differing = [n for n, h in comparable.items() if REFERENCE_EXPORT_SHA256.get(n) != h]
phase_check('P03-12', 'Output tables identical to the reference run (SHA-256)', not differing,
       f'{len(comparable) - len(differing)} of {len(comparable)} tables identical (2 run-specific tables not compared)' + (f"; differ: {', '.join(differing)}" if differing else ''), warn_only=True)

[PASS] P03-10: 674 of 674 checks passed (exported as validation_summary.csv)
[PASS] P03-11: 41 inputs unchanged; 13 tables and standardisation_run_manifest.json written to data/processed/standardisation_and_quality_checks/
[PASS] P03-12: 11 of 11 tables identical (2 run-specific tables not compared)


## 10. Validation summary

This phase's checks and the overall result. The 674 individual structural checks behind P03-10 are in `validation_summary.csv`. The table is also saved as `phase03_checks.csv`.

In [11]:
check_table = pd.DataFrame(phase_checks)
check_table.to_csv(OUT / "phase03_checks.csv", index=False, encoding="utf-8-sig", lineterminator="\n")
counts = check_table.status.value_counts()
n_pass, n_warn, n_fail = (int(counts.get(s, 0)) for s in ("PASS", "WARN", "FAIL"))
display(check_table)
print(f"Phase 03: {n_pass} of {len(check_table)} checks passed, {n_warn} warning(s), {n_fail} failure(s).")

,check_id,description,status,detail
0,P03-01,Project folder located,PASS,repository root found ('hydrogel-github-main')
1,P03-02,Phase 01 and Phase 02 manifests present,PASS,present
2,P03-03,"All inputs match their manifests, counts and schemas",PASS,41 files verified; 133 handoff checks passed
3,P03-04,Phase 02 tables identical to the reference run,PASS,26 of 26 tables identical
4,P03-05,Every source cell and image record matches the original files,PASS,"39,100 cells and 1,031 image records reconciled"
5,P03-06,Transformations applied only where supported,PASS,"applied 29,900; text_preserved 4,600; deferred_unit_semantics 4,600"
6,P03-07,Strain conversion matches the preview and reverses exactly,PASS,"100 percent cells divided by 100; 2,200 fraction cells kept"
7,P03-08,Image blanks unchanged and dates standardised,PASS,"1,031 of 1,031 dates standardised; completeness_sam 943 unknown; contain_sps 922 unknown"
8,P03-09,QC findings linked to exact cells and records,PASS,"25 findings (24 from Phase 02, 1 new); 2,302 cell/record links; 415 shear-rate cells below 0.01 1/s"
9,P03-10,Structural ledger,PASS,674 of 674 checks passed (exported as validation_summary.csv)


Phase 03: 12 of 12 checks passed, 0 warning(s), 0 failure(s).


## 11. Findings

Each row summarises one result with its status label, what it means, and whether it needs attention. Details are in the exported table named in the last column.

In [12]:
n_pct = int(strain.source_header.eq("ɣ in %").sum())
n_frac = int(strain.source_header.eq("ɣ in -").sum())
n_deferred = int(standardised.transformation_status.eq("deferred_unit_semantics").sum())
n_applied = int(standardised.transformation_status.eq("applied").sum())
n_neg = int((response & num.le(0)).sum())
n_axis = int(axis_bad.sum())
findings_table = pd.DataFrame([
    {"finding": f"All {verified_cells:,} source cells and {len(images):,} image records match the original files exactly",
     "status": "Verified", "meaning": "Phase 02 extraction is faithful; standardisation starts from the true source values",
     "attention": "No", "details": "validation_summary.csv"},
    {"finding": f"{n_applied:,} cells standardised; {n_pct} percentage-strain cells divided by 100; {n_frac:,} fraction-strain cells kept as labelled",
     "status": "Verified", "meaning": "Strain is now on one scale (fraction); the conversion reverses exactly",
     "attention": "No", "details": "measurement_values_standardised.csv"},
    {"finding": f"{n_deferred:,} viscosity cells (η and |η*|) keep the source token 'mPas'; no Pa·s value",
     "status": "Unresolved", "meaning": "Viscosity cannot yet be compared with values reported in Pa·s",
     "attention": "Yes — confirm the instrument's unit notation", "details": "standardisation_decisions.csv"},
    {"finding": f"{n_neg} non-positive response cells and {n_axis} non-positive shear-rate cells retained and flagged",
     "status": "Preliminary", "meaning": "Cause not established (possibly the measurement limit); excluded from nothing",
     "attention": "Yes — review before log scales or model fitting", "details": "qc_observation_links.csv"},
    {"finding": f"{len(below)} shear-rate cells below the paper's stated 0.01 s⁻¹ (new finding P3-QC-001)",
     "status": "Verified", "meaning": "The measured range extends below the published method; points retained",
     "attention": "No action in Phase 03", "details": "qc_findings.csv"},
    {"finding": f"{len(images):,} image dates converted to ISO format; blanks kept as unknown",
     "status": "Verified", "meaning": "Dates are consistent; no blank is read as 'no' or a failed print",
     "attention": "No", "details": "image_records_standardised.csv"},
    {"finding": "No linear viscoelastic region selected and no curve fitted",
     "status": "Assumption", "meaning": "Representative moduli are not derived; this is out of scope for standardisation",
     "attention": "No action in Phase 03", "details": "standardisation_decisions.csv"},
    {"finding": "Physical samples, formulations and replicates not assigned",
     "status": "Unresolved", "meaning": "Each point carries its worksheet key; Phase 04 tests candidate links",
     "attention": "No action in Phase 03", "details": "measurement_points_view.csv"},
])
findings_table

,finding,status,meaning,attention,details
0,"All 39,100 source cells and 1,031 image records match the original files exactly",Verified,Phase 02 extraction is faithful; standardisation starts from the true source values,No,validation_summary.csv
1,"29,900 cells standardised; 100 percentage-strain cells divided by 100; 2,200 fraction-strain cells kept as labelled",Verified,Strain is now on one scale (fraction); the conversion reverses exactly,No,measurement_values_standardised.csv
2,"4,600 viscosity cells (η and |η*|) keep the source token 'mPas'; no Pa·s value",Unresolved,Viscosity cannot yet be compared with values reported in Pa·s,Yes — confirm the instrument's unit notation,standardisation_decisions.csv
3,20 non-positive response cells and 2 non-positive shear-rate cells retained and flagged,Preliminary,Cause not established (possibly the measurement limit); excluded from nothing,Yes — review before log scales or model fitting,qc_observation_links.csv
4,415 shear-rate cells below the paper's stated 0.01 s⁻¹ (new finding P3-QC-001),Verified,The measured range extends below the published method; points retained,No action in Phase 03,qc_findings.csv
5,"1,031 image dates converted to ISO format; blanks kept as unknown",Verified,Dates are consistent; no blank is read as 'no' or a failed print,No,image_records_standardised.csv
6,No linear viscoelastic region selected and no curve fitted,Assumption,Representative moduli are not derived; this is out of scope for standardisation,No action in Phase 03,standardisation_decisions.csv
7,"Physical samples, formulations and replicates not assigned",Unresolved,Each point carries its worksheet key; Phase 04 tests candidate links,No action in Phase 03,measurement_points_view.csv


**Handoff to Phase 04.** The standardised cell table, point view, image records and QC links in `data/processed/standardisation_and_quality_checks/` are the inputs to Phase 04, which builds the record-linkage layer. Use `source_value_json`, `numeric_value` and `source_unit` for the original values; `standardised_value` is empty wherever a conversion is deferred.

**References.** [Zenodo record 19602891](https://zenodo.org/records/19602891) · Zhang, Elvitigala and Sakai (2026), [doi:10.1080/17452759.2026.2671497](https://doi.org/10.1080/17452759.2026.2671497), Section 5.1.